# 01 - Embed chunks and build the vector index

Runs on a Colab T4 GPU. Steps: download the chunks from the private Hugging Face dataset, measure the real
token length of every chunk (the chunker only estimates it), embed them with `BAAI/bge-m3`, build an exact
FAISS index, run a smoke test, and upload the embeddings back to the dataset repo.

**Before running:** Runtime > Change runtime type > T4 GPU, and add a Colab Secret named `HF_TOKEN`
(a Hugging Face token with write access) with "Notebook access" enabled.

In [ ]:
import os
import subprocess
import sys

REPO = "agentic-rag-eval"
BRANCH = "p1-agentic-rag"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://github.com/ilaria-cesa/{REPO}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random

import numpy as np
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

from src.retrieval.embed import load_chunks

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never hard-code the token
HF_REPO = "IlariaCesa/sec-10k-chunks"
chunks = load_chunks(hf_hub_download(HF_REPO, "chunks_1600.jsonl", repo_type="dataset"))
texts = [c["embed_text"] for c in chunks]
print(len(chunks), "chunks")

In [ ]:
from sentence_transformers import SentenceTransformer

from src.retrieval.embed import length_report, token_lengths

MODEL_NAME = "BAAI/bge-m3"
MAX_LEN = 640  # measured max chunk length is 633 tokens, so nothing is truncated
model = SentenceTransformer(MODEL_NAME, device="cuda", model_kwargs={"torch_dtype": torch.float16})
model.max_seq_length = MAX_LEN

lengths = token_lengths(texts, model.tokenizer)
report = length_report(lengths, MAX_LEN)
report["chars_per_token"] = round(sum(len(t) for t in texts) / sum(lengths), 2)
report["n_over_512"] = sum(n > 512 for n in lengths)  # what a 512 limit would have truncated
print(report)

In [ ]:
import importlib.metadata as md
import json

report["versions"] = {
    p: md.version(p)
    for p in [
        "sentence-transformers",
        "transformers",
        "torch",
        "faiss-cpu",
        "numpy",
        "huggingface_hub",
    ]
}
report["model"] = MODEL_NAME
os.makedirs("results", exist_ok=True)
with open("results/embedding_report.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

In [ ]:
from src.retrieval.embed import embed_texts

embeddings = embed_texts(model, texts, batch_size=32)
print(embeddings.shape, embeddings.dtype)

In [ ]:
from src.retrieval.index import build_index, search

index = build_index(embeddings)
smoke_queries = [
    "What are the main credit risks for JPMorgan Chase?",
    "How does Apple describe supply chain risks?",
    "How do catastrophe losses affect Travelers?",
]
for q in smoke_queries:  # smoke test only, NOT an evaluation
    scores, ids = search(index, embed_texts(model, [q]), k=3)
    print("\nQ:", q)
    for s, i in zip(scores[0], ids[0]):
        print(f"  {s:.3f}  {chunks[i]['chunk_id']}  {chunks[i]['text'][:110]!r}")

In [ ]:
from huggingface_hub import HfApi

np.save("embeddings_bge-m3_1600.npy", embeddings.astype(np.float16))
meta = {
    "model": MODEL_NAME,
    "max_seq_length": MAX_LEN,
    "chunks_file": "chunks_1600.jsonl",
    "n": len(chunks),
    "dim": int(embeddings.shape[1]),
    "dtype": "float16",
    "normalized": True,
    "first_chunk_ids": [c["chunk_id"] for c in chunks[:3]],
}
with open("embeddings_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
api = HfApi()
for name in ["embeddings_bge-m3_1600.npy", "embeddings_meta.json"]:
    api.upload_file(path_or_fileobj=name, path_in_repo=name, repo_id=HF_REPO, repo_type="dataset")
print("uploaded embeddings to", HF_REPO)

Next: copy the printed `report` (cell 5) back into the chat. `results/embedding_report.json` is the file
that goes into the repo.